# Sliding Window (Kỹ thuật Cửa Sổ Trượt)

## 1. Định nghĩa

**Sliding Window** là kỹ thuật duy trì một "cửa sổ" — đoạn con liên tiếp trong mảng/chuỗi — và trượt cửa sổ đó qua dữ liệu thay vì tính lại từ đầu mỗi lần.

Cửa sổ được định nghĩa bởi **hai con trỏ**:
- `left` — điểm bắt đầu cửa sổ
- `right` — điểm kết thúc cửa sổ

### Mục tiêu chính
| Tiêu chí | Brute Force | Sliding Window |
|---|---|---|
| Time Complexity | O(n²) | **O(n)** |
| Space Complexity | O(1) ~ O(n) | **O(1) ~ O(k)** |

> *"Điều này cho phép chúng ta duyệt qua dữ liệu chỉ một lần với hai con trỏ left và right, giảm độ phức tạp tính toán xuống còn O(n)."*

## 2. Hai dạng Sliding Window

### Dạng 1: Fixed-Size Window (Cửa sổ cố định)
Kích thước cửa sổ **không đổi** = k trong suốt quá trình.

```
[2, 1, 5, 1, 3, 2]   k=3
 [2  1  5] 1  3  2   sum=8
  2 [1  5  1] 3  2   sum=7  → trượt: bỏ arr[left], thêm arr[right]
  2  1 [5  1  3] 2   sum=9  ← max
  2  1  5 [1  3  2]  sum=6
```

---

### Dạng 2: Dynamic-Size Window (Cửa sổ động)
Kích thước cửa sổ **thay đổi** — mở rộng hoặc thu hẹp theo điều kiện bài toán.

```
"abcabcbb"
 [a]               → không trùng, right++
 [a b]             → không trùng, right++
 [a b c]           → không trùng, right++
 [a b c] a         → 'a' trùng! left++ để thu hẹp
   [b c a]         → không trùng, right++
```

## 3. Khi nào nên dùng Sliding Window?

✅ Tìm **subarray/substring** thỏa điều kiện (tổng, trung bình, ký tự)  
✅ Tìm đoạn con **ngắn nhất / dài nhất** thỏa điều kiện  
✅ Bài toán yêu cầu xử lý **dữ liệu liên tiếp** (contiguous)  
✅ Khi brute force dùng **2 vòng lặp lồng nhau** → thường tối ưu được bằng sliding window

## 4. Ví dụ thực tế

### Ví dụ 1: Tổng lớn nhất của subarray kích thước k (Fixed Window)

In [ ]:
def max_sum_subarray(arr: list[int], k: int) -> int | str:
    """Tìm tổng lớn nhất của subarray có đúng k phần tử.

    Args:
        arr: Danh sách số nguyên.
        k: Kích thước cửa sổ.

    Returns:
        Tổng lớn nhất tìm được.
    """
    n = len(arr)
    if k > n:
        raise ValueError(f"k={k} lớn hơn độ dài mảng n={n}")

    # Tính tổng cửa sổ đầu tiên
    current_sum = sum(arr[:k])
    max_sum = current_sum
    left = 0

    for right in range(k, n):
        current_sum += arr[right]   # thêm phần tử mới vào phải
        current_sum -= arr[left]    # bỏ phần tử cũ ở trái
        left += 1
        max_sum = max(max_sum, current_sum)

    return max_sum


# Test
arr = [2, 1, 5, 1, 3, 2]
print(max_sum_subarray(arr, k=3))  # 9  → subarray [5,1,3]

**Tại sao O(n)?** Mỗi lần trượt chỉ cần **1 phép cộng + 1 phép trừ**, không tính lại toàn bộ cửa sổ:
```
Brute force: tính sum(arr[i:i+k]) mỗi lần → O(k) mỗi bước → O(n*k)
Sliding window: current_sum += arr[right] - arr[left]   → O(1) mỗi bước → O(n)
```

### Ví dụ 2: Chuỗi con dài nhất không có ký tự lặp (Dynamic Window)

In [ ]:
def longest_unique_substring(s: str) -> int:
    """Tìm độ dài chuỗi con dài nhất không có ký tự lặp.

    Args:
        s: Chuỗi đầu vào.

    Returns:
        Độ dài chuỗi con dài nhất.
    """
    if not s:
        return 0

    char_set = set()  # các ký tự trong cửa sổ hiện tại
    left = 0
    max_length = 0

    for right in range(len(s)):
        # Thu hẹp cửa sổ cho đến khi s[right] không trùng
        while s[right] in char_set:
            char_set.remove(s[left])
            left += 1

        char_set.add(s[right])
        max_length = max(max_length, right - left + 1)

    return max_length


# Test
print(longest_unique_substring("abcabcbb"))  # 3 → "abc"
print(longest_unique_substring("bbbbb"))     # 1 → "b"
print(longest_unique_substring("pwwkew"))    # 3 → "wke"

**Trace với `"abcabcbb"`:**
```
right=0 'a': set={a}       window="a"     len=1
right=1 'b': set={a,b}     window="ab"    len=2
right=2 'c': set={a,b,c}   window="abc"   len=3 ← max
right=3 'a': 'a' trùng! → xóa s[left]='a', left=1
             set={b,c,a}   window="bca"   len=3
right=4 'b': 'b' trùng! → xóa s[left]='b', left=2
             set={c,a,b}   window="cab"   len=3
...
```

### Ví dụ 3: Subarray ngắn nhất có tổng ≥ target (Dynamic Window)

In [ ]:
import math


def min_subarray_len(target: int, nums: list[int]) -> int:
    """Tìm độ dài subarray ngắn nhất có tổng >= target.

    Args:
        target: Giá trị tổng tối thiểu cần đạt.
        nums: Danh sách số nguyên dương.

    Returns:
        Độ dài nhỏ nhất, hoặc 0 nếu không tồn tại.
    """
    min_length = math.inf
    current_sum = 0
    left = 0

    for right in range(len(nums)):
        current_sum += nums[right]  # mở rộng cửa sổ sang phải

        # Thu hẹp từ trái chừng nào tổng vẫn >= target
        while current_sum >= target:
            min_length = min(min_length, right - left + 1)
            current_sum -= nums[left]
            left += 1

    return min_length if min_length != math.inf else 0


# Test
print(min_subarray_len(7, [2, 3, 1, 2, 4, 3]))  # 2 → [4,3]
print(min_subarray_len(4, [1, 4, 4]))            # 1 → [4]
print(min_subarray_len(11, [1, 1, 1, 1, 1]))     # 0 → không tồn tại

**Trace với `target=7, nums=[2,3,1,2,4,3]`:**
```
right=0: sum=2  < 7  → mở rộng
right=1: sum=5  < 7  → mở rộng
right=2: sum=6  < 7  → mở rộng
right=3: sum=8  ≥ 7  → min_len=4, left++, sum=6  < 7  → dừng thu hẹp
right=4: sum=10 ≥ 7  → min_len=3, left++, sum=7  ≥ 7
                      → min_len=2, left++, sum=6  < 7  → dừng
right=5: sum=9  ≥ 7  → min_len=2, left++, sum=7  ≥ 7
                      → min_len=2, left++, sum=3  < 7  → dừng
Kết quả: 2
```

## 5. Template tổng quát

In [ ]:
# Template 1: Fixed-Size Window
def fixed_window(arr: list, k: int):
    # Khởi tạo cửa sổ đầu tiên
    window = ...  # tính giá trị cho arr[0:k]
    result = window

    for right in range(k, len(arr)):
        window += arr[right]        # thêm phần tử mới
        window -= arr[right - k]    # bỏ phần tử cũ (left = right - k)
        result = max(result, window)  # hoặc min tùy bài

    return result


# Template 2: Dynamic-Size Window
def dynamic_window(arr: list):
    left = 0
    result = 0
    window_state = ...  # set, dict, int tùy bài

    for right in range(len(arr)):
        # Mở rộng: thêm arr[right] vào window
        # ...

        # Thu hẹp: khi window vi phạm điều kiện
        while ...  # điều kiện vi phạm:
            # bỏ arr[left] khỏi window
            left += 1

        # Cập nhật kết quả
        result = max(result, right - left + 1)

    return result

## 6. Phân tích độ phức tạp

| Bài toán | Dạng | Time | Space |
|---|---|---|---|
| Max sum subarray size k | Fixed | O(n) | O(1) |
| Longest unique substring | Dynamic | O(n) | O(min(m,n)) |
| Min subarray sum ≥ target | Dynamic | O(n) | O(1) |

> **Tại sao Dynamic Window vẫn là O(n)?**  
> Mỗi phần tử được thêm vào cửa sổ **đúng 1 lần** và bỏ ra **đúng 1 lần** → tổng thao tác = 2n → O(n).

## 7. So sánh Sliding Window vs Two Pointers

| | Two Pointers | Sliding Window |
|---|---|---|
| Dữ liệu | Thường cần sorted | Không cần sorted |
| Mục tiêu | Tìm cặp/bộ thỏa điều kiện | Tìm đoạn con liên tiếp |
| Con trỏ | Di chuyển ngược chiều hoặc cùng chiều | Luôn cùng chiều (left ≤ right) |
| Ví dụ điển hình | Two Sum, 3Sum, palindrome | Max sum, longest substring |

> Sliding Window thực chất là **biến thể của Two Pointers** chuyên dùng cho bài toán đoạn con liên tiếp.

## 8. Anti-patterns cần tránh

| Lỗi thường gặp | Cách tránh |
|---|---|
| Tính lại toàn bộ window mỗi bước | Dùng biến `current_sum` cập nhật tăng dần |
| Dùng Fixed Window cho bài cần Dynamic | Xác định rõ kích thước có cố định không |
| Quên thu hẹp cửa sổ khi vi phạm điều kiện | Luôn có vòng `while` kiểm tra sau khi mở rộng |
| Index out of range ở Fixed Window | Khởi tạo cửa sổ đầu tiên `arr[:k]` trước vòng lặp |

## 9. Bài tập thực hành

| # | Bài toán | Dạng | Độ khó |
|---|---|---|---|
| 1 | Maximum Average Subarray I | Fixed | Easy |
| 2 | Longest Substring Without Repeating Characters | Dynamic | Medium |
| 3 | Minimum Size Subarray Sum | Dynamic | Medium |
| 4 | Longest Substring with At Most K Distinct Characters | Dynamic | Medium |
| 5 | Sliding Window Maximum (dùng deque) | Fixed | Hard |

## 10. Bài tập: Maximum Average Subarray

In [ ]:
def find_max_average(nums: list[int], k: int) -> float:
    """Tìm giá trị trung bình lớn nhất của subarray kích thước k.

    Args:
        nums: Danh sách số nguyên.
        k: Kích thước cửa sổ.

    Returns:
        Giá trị trung bình lớn nhất.
    """
    current_sum = sum(nums[:k])
    max_sum = current_sum

    for right in range(k, len(nums)):
        current_sum += nums[right] - nums[right - k]  # thêm phải, bỏ trái
        max_sum = max(max_sum, current_sum)

    return max_sum / k


# Test
print(find_max_average([1, 12, -5, -6, 50, 3], k=4))  # 12.75 → [12,-5,-6,50]
print(find_max_average([5], k=1))                      # 5.0